# 98. Final dataset assembly

Bilingual clean-constraints patch, main generation loop, zero-gold generation, output writing, and summary.

In [ ]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires the nbformat package.
from pathlib import Path
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())


<a id="v2-clean-bilingual-patch"></a>
## v2-clean bilingual patch
Normalize constraints, generate English queries, and fetch English gold labels before saving examples.


In [ ]:

# ============================================================
# v2-clean patch: clean constraints + English queries + English gold labels
# ============================================================
# This cell is intentionally post-generation: it keeps the original rich
# Russian template generators, but normalizes every produced example before
# it is written to JSONL.

SERVICE_CONSTRAINT_KEYS = {
    "source_dataset", "source_file", "template_id", "template_family", "type",
    "qid", "qids", "sparql_query", "ask_validator_sparql", "date_properties",
    "text_clues", "full_constraints", "created_at", "gold_answer_qids",
    "gold_answer_labels_ru", "gold_answer_labels_en", "is_advanced",
    "_score_min", "score", "rating", "movie_length_range", "year_range",
    "method", "machine_evidence", "old_constraints_hash", "new_constraints_hash",
    "gold_enrichment_status", "gold_enrichment_method", "gold_enrichment_meta",
}
SERVICE_PREFIXES = ("gold_", "template_")
SERVICE_SUFFIXES = ("_qid", "_qids")

RU_TO_EN_VALUE = {
    # countries
    "США": "United States of America",
    "Соединённые Штаты Америки": "United States of America",
    "Соединенные Штаты Америки": "United States of America",
    "Великобритания": "United Kingdom",
    "Франция": "France",
    "Германия": "Germany",
    "Италия": "Italy",
    "Индия": "India",
    "Япония": "Japan",
    "Россия": "Russia",
    "Канада": "Canada",
    "Испания": "Spain",
    "Китай": "China",
    "Бразилия": "Brazil",
    "Австралия": "Australia",
    # continents / regions
    "Африка": "Africa",
    "Европа": "Europe",
    "Азия": "Asia",
    "Северная Америка": "North America",
    "Южная Америка": "South America",
    "Океания": "Oceania",
    # common languages
    "английский язык": "English",
    "французский язык": "French",
    "арабский язык": "Arabic",
    "испанский язык": "Spanish",
    "русский язык": "Russian",
    "немецкий язык": "German",
    "португальский язык": "Portuguese",
    # currencies
    "евро": "euro",
    "доллар США": "United States dollar",
    "фунт стерлингов": "pound sterling",
    "центральноафриканский франк КФА": "Central African CFA franc",
    "восточноафриканский шиллинг": "East African shilling",
    # cinema genres
    "драма": "drama film",
    "фильм-драма": "drama film",
    "боевик": "action film",
    "триллер": "thriller film",
    "фильм ужасов": "horror film",
    "ужасы": "horror film",
    "криминальный фильм": "crime film",
    "комедия": "comedy film",
    "фантастика": "science fiction film",
    "научная фантастика": "science fiction film",
    "мелодрама": "romance film",
    "анимационный фильм": "animated film",
    "мультфильм": "animated film",
    # sports
    "футбол": "association football",
    "Чемпионат Германии по футболу": "Bundesliga",
    # ingredients / dishes
    "мука": "flour",
    "говядина": "beef",
    "свинина": "pork",
    "курица": "chicken",
    "сыр": "cheese",
    "яйцо": "egg",
    "яйца": "egg",
    "хлеб": "bread",
    "мексиканская кухня": "Mexican cuisine",
    "итальянская кухня": "Italian cuisine",
    "французская кухня": "French cuisine",
    "индийская кухня": "Indian cuisine",
    "китайская кухня": "Chinese cuisine",
    "японская кухня": "Japanese cuisine",
}

KEY_RENAMES = {
    "genre_ru": "genre",
    "country_ru": "country",
    "region_ru": "region",
    "continent_ru": "continent",
    "official_language_ru": "official_language",
    "currency_ru": "currency",
    "actor_ru": "actor",
    "director_ru": "director",
    "author_ru": "author",
    "artist_ru": "artist",
    "manufacturer_ru": "manufacturer",
    "operator_ru": "operator",
    "league_ru": "league",
    "sport_ru": "sport",
    "city_ru": "city",
    "collection_ru": "collection",
    "movement_ru": "movement",
    "occupation_ru": "occupation",
    "citizenship_ru": "citizenship",
    "award_ru": "award",
    "developer_ru": "developer",
    "operating_system_ru": "operating_system",
    "programming_language_ru": "programming_language",
    "license_ru": "license",
    "cuisine_ru": "cuisine",
    "dish_type_ru": "dish_type",
    "seed_ru": "seed",
    "franchise_ru": "franchise",
    "include_ingredients_ru": "include_ingredients",
    "exclude_ingredients_ru": "exclude_ingredients",
}

KIND_RU_TO_EN = {
    "фильм": "film",
    "сериал": "series",
    "книга": "book",
    "страна": "country",
    "блюдо": "dish",
    "аэропорт": "airport",
    "смартфон": "smartphone",
    "программное обеспечение": "software",
    "картина": "painting",
    "человек": "person",
    "спортивный клуб": "sports_club",
    "космический аппарат": "spacecraft",
}

DOMAIN_DEFAULT_KIND = {
    "cinema": "film",
    "books": "book",
    "countries": "country",
    "dishes": "dish",
    "spacecraft": "spacecraft",
    "sports_clubs": "sports_club",
    "smartphones": "smartphone",
    "airports": "airport",
    "software": "software",
    "paintings": "painting",
    "museums": "museum",
    "people": "person",
    "scientists": "person",
    "physicists": "person",
    "mathematicians": "person",
    "videogames": "video_game",
    "music_albums": "music_album",
    "universities": "university",
    "cars": "car",
    "geo": "geographic_object",
    "geo_ru": "geographic_object",
}

def _strip_quotes_ru(s: Any) -> str:
    s = str(s).strip()
    if len(s) >= 2 and ((s[0] == "«" and s[-1] == "»") or (s[0] == '"' and s[-1] == '"')):
        return s[1:-1].strip()
    return s

def _translate_scalar_value(v: Any, qid: Optional[str] = None) -> Any:
    if isinstance(v, (int, float, bool)) or v is None:
        return v
    if isinstance(v, list):
        return [_translate_scalar_value(x) for x in v if x not in (None, "", [], {})]
    s = _strip_quotes_ru(v)
    if qid:
        lbl = wd_label_en(qid)
        if lbl:
            return lbl
    return RU_TO_EN_VALUE.get(s, s)

def _entity_map_for_qids(qids: List[str]) -> Dict[str, dict]:
    qids = [q for q in dict.fromkeys(qids or []) if isinstance(q, str) and re.fullmatch(r"Q\d+", q)]
    if not qids:
        return {}
    # wd_get_entities_ru already requests ru|en and caches entities per QID.
    try:
        return wd_get_entities_ru(qids)
    except Exception:
        pass

    # fallback
    out = {}
    for chunk in _chunks(qids, 50):
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "labels|aliases|sitelinks",
            "languages": "en|ru",
            "format": "json",
        }
        try:
            r = requests.get(WIKI_API, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            out.update((data or {}).get("entities", {}) or {})
        except Exception:
            continue
    return out

def wd_label_en(qid: Optional[str], fallback: str = "") -> str:
    if not qid or not isinstance(qid, str) or not re.fullmatch(r"Q\d+", qid):
        return fallback or ""
    key = f"label_en_{qid}"
    try:
        cached = wd.cache.get(key)
        if cached and cached.get("label"):
            return cached["label"]
    except Exception:
        pass
    ent = _entity_map_for_qids([qid]).get(qid, {})
    label = (((ent.get("labels") or {}).get("en") or {}).get("value") or
             ((ent.get("labels") or {}).get("ru") or {}).get("value") or
             fallback or qid)
    try:
        wd.cache.set(key, {"label": label})
    except Exception:
        pass
    return label

def wd_labels_en(qids: List[str], fallback_labels: Optional[List[str]] = None) -> List[str]:
    qids = list(qids or [])
    fallback_labels = list(fallback_labels or [])
    ent_map = _entity_map_for_qids(qids)
    out = []
    for i, q in enumerate(qids):
        fallback = fallback_labels[i] if i < len(fallback_labels) else ""
        ent = ent_map.get(q, {})
        label = (((ent.get("labels") or {}).get("en") or {}).get("value") or
                 ((ent.get("labels") or {}).get("ru") or {}).get("value") or
                 fallback or q)
        out.append(str(label))
    return out

def _paired_qid_for_key(c: Dict[str, Any], key: str) -> Optional[str]:
    # genre_ru -> genre_qid; country_ru -> country_qid etc.
    base = key[:-3] if key.endswith("_ru") else key
    for cand in (f"{base}_qid", f"{key}_qid", "seed_qid"):
        q = c.get(cand)
        if isinstance(q, str) and re.fullmatch(r"Q\d+", q):
            return q
    return None

def _normalize_rating_fields(c: Dict[str, Any], out: Dict[str, Any]) -> None:
    # rating: ["imdb"|"kp", ">=", 7.5] or _score_min from old generators
    rating = c.get("rating")
    if isinstance(rating, (list, tuple)) and len(rating) >= 3:
        src = str(rating[0]).lower()
        op = str(rating[1])
        try:
            val = float(rating[2])
        except Exception:
            val = None
        if val is not None:
            if src in ("kp", "kinopoisk", "кинопоиск"):
                out["rating_kinopoisk_min" if ">=" in op or ">" in op else "rating_kinopoisk_max"] = val
            elif src in ("imdb",):
                out["rating_imdb_min" if ">=" in op or ">" in op else "rating_imdb_max"] = val
            else:
                out["rating_min" if ">=" in op or ">" in op else "rating_max"] = val
    if "_score_min" in c and "rating_kinopoisk_min" not in out and "rating_imdb_min" not in out:
        # Use text later to decide source; default generic.
        try:
            out["rating_min"] = float(c["_score_min"])
        except Exception:
            pass

def _normalize_movie_length(c: Dict[str, Any], out: Dict[str, Any]) -> None:
    rng = c.get("movie_length_range")
    if isinstance(rng, (list, tuple)) and len(rng) == 2:
        try:
            a, b = int(rng[0]), int(rng[1])
            if a > 1:
                out["movie_length_min_minutes"] = a
            if b < 1000:
                out["movie_length_max_minutes"] = b
        except Exception:
            pass

def _extract_years_from_text(text: str) -> Optional[Tuple[int, int]]:
    q = text or ""
    # 2018–2025, 2018-2025
    m = re.search(r'(?<!\d)(19\d{2}|20\d{2})\s*[–—-]\s*(19\d{2}|20\d{2})(?!\d)', q)
    if m:
        return int(m.group(1)), int(m.group(2))
    # с 2018 по 2025
    m = re.search(r'с\s+(19\d{2}|20\d{2})\s+по\s+(19\d{2}|20\d{2})', q, flags=re.I)
    if m:
        return int(m.group(1)), int(m.group(2))
    # 2015 года
    m = re.search(r'(?<!\d)(19\d{2}|20\d{2})\s*г(?:ода|\.|г)?', q, flags=re.I)
    if m:
        y = int(m.group(1))
        return y, y
    return None

def _extract_population_from_text(text: str) -> Optional[Tuple[int, int]]:
    q = text or ""
    if "населен" not in q.lower() and "насел" not in q.lower() and "числен" not in q.lower():
        return None
    nums = re.findall(r'\d[\d\s\u00A0]*', q)
    vals = []
    for n in nums:
        try:
            vals.append(int(re.sub(r'\D', '', n)))
        except Exception:
            pass
    vals = [v for v in vals if v >= 1_000]  # avoid requested_count
    if len(vals) >= 2:
        return vals[0], vals[1]
    return None

def _extract_rating_from_text(text: str, out: Dict[str, Any]) -> None:
    q = text or ""
    low = q.lower()
    # value next to rating wording
    m = re.search(r'(IMDb|IMDB|imdb|Кинопоиск|кинопоиск)[^0-9]{0,40}(\d+(?:[.,]\d+)?)', q)
    if not m:
        return
    src = m.group(1).lower()
    val = float(m.group(2).replace(",", "."))
    is_max = any(x in low for x in ["не выше", "не более", "до ", "меньше", "ниже"])
    if "imdb" in src:
        out["rating_imdb_max" if is_max else "rating_imdb_min"] = val
        out.pop("rating_min", None)
    elif "кинопоиск" in src:
        out["rating_kinopoisk_max" if is_max else "rating_kinopoisk_min"] = val
        out.pop("rating_min", None)

def _extract_movie_length_from_text(text: str, out: Dict[str, Any]) -> None:
    q = (text or "").lower()
    m = re.search(r'не\s+более\s+(\d+(?:[.,]\d+)?)\s*час', q)
    if m:
        out["movie_length_max_minutes"] = int(float(m.group(1).replace(",", ".")) * 60)
    m = re.search(r'более\s+(\d+(?:[.,]\d+)?)\s*час', q)
    if m and "не более" not in q[max(0, m.start()-5):m.end()+5]:
        out["movie_length_min_minutes"] = int(float(m.group(1).replace(",", ".")) * 60)
    m = re.search(r'от\s+(\d+)\s+до\s+(\d+)\s+мин', q)
    if m:
        out["movie_length_min_minutes"] = int(m.group(1))
        out["movie_length_max_minutes"] = int(m.group(2))

def _extract_quoted_values(text: str) -> List[str]:
    return [m.group(1).strip() for m in re.finditer(r'«([^»]+)»', text or "")]

def _augment_constraints_from_text(domain: str, text: str, out: Dict[str, Any]) -> None:
    q = text or ""
    low = q.lower()

    # default kind
    out.setdefault("kind", DOMAIN_DEFAULT_KIND.get(domain, out.get("kind")))

    yrs = _extract_years_from_text(q)
    if yrs and "year_from" not in out and not any(k.endswith("_year_from") for k in out):
        out["year_from"], out["year_to"] = yrs

    pop = _extract_population_from_text(q)
    if pop:
        out["population_min"], out["population_max"] = pop

    # negative / type filters
    if "франшиз" in low or "серии" in low:
        if any(x in low for x in ["не является", "не являются", "не входит", "не входят", "не частью", "не часть"]):
            out["not_franchise"] = True
    if "не сери" in low:
        out["not_series"] = True
    if "не мульт" in low:
        out["not_animation"] = True
    if "не аниме" in low:
        out["not_anime"] = True
    if "полнометраж" in low:
        out["full_length_only"] = True
    if "которых уже не существует" in low or "уже не существует" in low:
        out["historical_only"] = True
        out["exists_now"] = False
    if "современные государства" in low or "существуют сейчас" in low:
        out["exists_now"] = True

    _extract_rating_from_text(q, out)
    _extract_movie_length_from_text(q, out)

    # Ingredients from dishes text
    if domain == "dishes":
        inc = []
        exc = []
        # quoted after "есть/содерж"
        for val in _extract_quoted_values(q):
            val_en = _translate_scalar_value(val)
            # very simple phrase window
            pos = q.find(f"«{val}»")
            window = low[max(0, pos-40):pos+len(val)+40]
            if any(x in window for x in ["нет", "без", "не содержит", "исключ"]):
                exc.append(val_en)
            elif any(x in window for x in ["есть", "содерж", "ингредиент"]):
                inc.append(val_en)
        if inc:
            out["include_ingredients"] = list(dict.fromkeys(out.get("include_ingredients", []) + inc))
        if exc:
            out["exclude_ingredients"] = list(dict.fromkeys(out.get("exclude_ingredients", []) + exc))

    # Common quoted direct fields when old constraints are missing.
    quoted = _extract_quoted_values(q)
    if domain == "cinema":
        if "жанр" in low and quoted and "genre" not in out:
            out["genre"] = _translate_scalar_value(quoted[0])
    if domain == "countries":
        if "официальный язык" in low and quoted:
            for v in quoted:
                if "язык" in v:
                    out.setdefault("official_language", _translate_scalar_value(v))
        if "валют" in low and quoted:
            out.setdefault("currency", _translate_scalar_value(quoted[-1]))
        if ("регион" in low or "континент" in low) and quoted:
            # if query says континент prefer continent, else region
            key = "continent" if "континент" in low else "region"
            out.setdefault(key, _translate_scalar_value(quoted[0]))
    if domain == "spacecraft":
        if "той же организац" in low or "тем же производител" in low:
            if quoted:
                out.setdefault("manufacturer_same_as_object", _translate_scalar_value(quoted[0]))
        if "тем же оператор" in low or "той же оператор" in low or "с тем же оператор" in low:
            if quoted:
                out.setdefault("operator_same_as_object", _translate_scalar_value(quoted[0]))
    if domain == "sports_clubs":
        if "той же лиг" in low and quoted:
            out.setdefault("league_same_as_object", _translate_scalar_value(quoted[0]))

def clean_constraints_for_example(ex: BenchmarkExample) -> Dict[str, Any]:
    old = dict(ex.constraints or {})
    out: Dict[str, Any] = {}

    # Normalize known composite fields first.
    _normalize_rating_fields(old, out)
    _normalize_movie_length(old, out)

    # Copy meaningful fields, drop service/QID/null/false.
    for key, val in old.items():
        if key in SERVICE_CONSTRAINT_KEYS:
            continue
        if key.startswith(SERVICE_PREFIXES) or key.endswith(SERVICE_SUFFIXES):
            continue
        if val is None or val == "" or val == [] or val == {}:
            continue
        if val is False:
            continue

        new_key = KEY_RENAMES.get(key, key)
        if new_key.endswith("_ru"):
            new_key = new_key[:-3]

        # Normalize old kind values.
        qid = _paired_qid_for_key(old, key)
        if new_key == "kind":
            if isinstance(val, str):
                out[new_key] = KIND_RU_TO_EN.get(val, val)
            else:
                out[new_key] = val
            continue

        out[new_key] = _translate_scalar_value(val, qid=qid)

    _augment_constraints_from_text(ex.domain, ex.query_text_ru, out)

    # Remove accidental false/null/service after augmentation.
    final = {}
    for k, v in out.items():
        if k in SERVICE_CONSTRAINT_KEYS or k.startswith(SERVICE_PREFIXES) or k.endswith(SERVICE_SUFFIXES):
            continue
        if v is None or v == "" or v == [] or v == {}:
            continue
        if v is False and k not in ("exists_now",):
            continue
        # no *_ru in clean English constraints
        if k.endswith("_ru"):
            k = k[:-3]
        final[k] = v

    # Stable key order.
    order = [
        "kind", "genre", "country", "continent", "region", "official_language", "currency",
        "year_from", "year_to", "publication_year_from", "publication_year_to",
        "creation_year_from", "creation_year_to", "release_year_from", "release_year_to",
        "birth_year_from", "birth_year_to", "population_min", "population_max",
        "actor", "director", "author", "artist", "manufacturer", "operator",
        "manufacturer_same_as_object", "operator_same_as_object", "league_same_as_object",
        "sport", "league", "city", "collection", "cuisine", "include_ingredients",
        "exclude_ingredients", "rating_imdb_min", "rating_imdb_max",
        "rating_kinopoisk_min", "rating_kinopoisk_max", "rating_min", "rating_max",
        "movie_length_min_minutes", "movie_length_max_minutes",
        "full_length_only", "not_franchise", "not_series", "not_tv_movie", "not_short_film",
        "not_animation", "not_anime", "historical_only", "exists_now",
    ]
    return {k: final[k] for k in order if k in final} | {k: v for k, v in final.items() if k not in order}

def _fmt_list(xs):
    return ", ".join(f'"{x}"' for x in xs)

def _period_phrase(c, generic="from {a} to {b}"):
    for a_key,b_key,label in [
        ("year_from","year_to","from {a} to {b}"),
        ("publication_year_from","publication_year_to","published from {a} to {b}"),
        ("creation_year_from","creation_year_to","created from {a} to {b}"),
        ("release_year_from","release_year_to","released from {a} to {b}"),
        ("birth_year_from","birth_year_to","born from {a} to {b}"),
    ]:
        if a_key in c and b_key in c:
            a,b=c[a_key],c[b_key]
            return f"in {a}" if a == b else label.format(a=a,b=b)
    return ""

def english_query_from_constraints(ex: BenchmarkExample, c: Dict[str, Any]) -> str:
    n = int(ex.requested_count or 5)
    domain = ex.domain
    kind = c.get("kind") or DOMAIN_DEFAULT_KIND.get(domain, "items")
    plural = {
        "film": "films", "country": "countries", "dish": "dishes", "spacecraft": "spacecraft",
        "sports_club": "sports clubs", "book": "books", "painting": "paintings", "person": "people",
        "airport": "airports", "software": "software items", "smartphone": "smartphones",
        "video_game": "video games", "music_album": "music albums", "university": "universities",
        "car": "car models", "geographic_object": "geographic objects",
    }.get(kind, str(kind).replace("_", " ") + "s")

    parts = []
    if domain == "cinema":
        base = f"Name {n} {plural}"
        if c.get("country_same_as_object"):
            parts.append(f'with the same country of origin as "{c["country_same_as_object"]}"')
        if c.get("genre"): parts.append(f'with genre "{c["genre"]}"')
        if c.get("country"): parts.append(f'from country {c["country"]}')
        if c.get("actor"): parts.append(f'featuring actor {c["actor"]}')
        if c.get("director"): parts.append(f'directed by {c["director"]}')
        p = _period_phrase(c)
        if p: parts.append(p)
        if c.get("rating_imdb_min") is not None: parts.append(f'with IMDb rating at least {c["rating_imdb_min"]}')
        if c.get("rating_kinopoisk_min") is not None: parts.append(f'with Kinopoisk rating at least {c["rating_kinopoisk_min"]}')
        if c.get("movie_length_max_minutes") is not None: parts.append(f'with runtime no more than {c["movie_length_max_minutes"]} minutes')
        if c.get("not_franchise"): parts.append("not part of a franchise or series")
        if c.get("full_length_only"): parts.append("only regular feature films")
        if c.get("not_series"): parts.append("not series")
        if c.get("not_tv_movie"): parts.append("not TV movies")
        if c.get("not_short_film"): parts.append("not short films")
        if c.get("not_animation"): parts.append("not animated films")
        return base + (", " + ", ".join(parts) if parts else "") + "."

    if domain == "countries":
        base = f"Find {n} countries"
        if c.get("continent"): parts.append(f'on continent "{c["continent"]}"')
        if c.get("region"): parts.append(f'in region "{c["region"]}"')
        if c.get("exists_now") is True: parts.append("only modern states that currently exist")
        if c.get("historical_only"): parts.append("only historical states that no longer exist")
        if c.get("official_language"): parts.append(f'where the official language is "{c["official_language"]}"')
        if c.get("currency"): parts.append(f'where the currency is "{c["currency"]}"')
        if c.get("population_min") is not None and c.get("population_max") is not None:
            parts.append(f'with population approximately between {c["population_min"]:,} and {c["population_max"]:,}')
        return base + (", " + ", ".join(parts) if parts else "") + "."

    if domain == "dishes":
        base = f"Find {n} dishes"
        if c.get("include_ingredients"): parts.append(f'that contain {_fmt_list(c["include_ingredients"])}')
        if c.get("exclude_ingredients"): parts.append(f'that do not contain {_fmt_list(c["exclude_ingredients"])}')
        if c.get("cuisine"): parts.append(f'from "{c["cuisine"]}" cuisine')
        return base + (", " + ", ".join(parts) if parts else "") + "."

    if domain == "spacecraft":
        base = f"Name {n} spacecraft"
        if c.get("manufacturer_same_as_object"): parts.append(f'manufactured by the same organization as "{c["manufacturer_same_as_object"]}"')
        if c.get("operator_same_as_object"): parts.append(f'with the same operator as "{c["operator_same_as_object"]}"')
        if c.get("manufacturer"): parts.append(f'manufactured by "{c["manufacturer"]}"')
        if c.get("operator"): parts.append(f'operated by "{c["operator"]}"')
        p = _period_phrase(c)
        if p: parts.append(f'launched {p}' if not p.startswith("in ") else f'launched {p}')
        return base + (", " + ", ".join(parts) if parts else "") + "."

    if domain == "sports_clubs":
        base = f"Name {n} sports clubs"
        if c.get("sport"): parts.append(f'in the sport: {c["sport"]}')
        if c.get("country"): parts.append(f'in country: {c["country"]}')
        if c.get("league"): parts.append(f'in league: {c["league"]}')
        if c.get("league_same_as_object"): parts.append(f'that play in the same league as "{c["league_same_as_object"]}"')
        if c.get("founded_year_from") and c.get("founded_year_to"):
            parts.append(f'founded between {c["founded_year_from"]} and {c["founded_year_to"]}')
        return base + (", " + ", ".join(parts) if parts else "") + "."

    # Generic fallback.
    base = f"Name {n} {plural}"
    simple_keys = [
        "genre", "country", "continent", "region", "official_language", "currency", "author", "artist",
        "developer", "manufacturer", "operator", "collection", "occupation", "citizenship", "award",
        "operating_system", "programming_language", "license", "city",
    ]
    for k in simple_keys:
        if c.get(k):
            parts.append(f'with {k.replace("_", " ")} "{c[k]}"')
    p = _period_phrase(c)
    if p: parts.append(p)
    return base + (", " + ", ".join(parts) if parts else "") + "."

def finalize_bilingual_clean_example(ex: BenchmarkExample) -> BenchmarkExample:
    # Clean constraints first; English query is generated from the cleaned constraints.
    clean = clean_constraints_for_example(ex)
    ex.constraints = clean
    ex.query_text_en = english_query_from_constraints(ex, clean)
    ex.gold_answer_labels_en = wd_labels_en(ex.gold_answer_qids, fallback_labels=ex.gold_answer_labels_ru)
    return ex

print("v2-clean patch loaded: clean constraints + query_text_en + gold_answer_labels_en")


<a id="final-dataset-assembly"></a>

## 25. Final dataset assembly

Append-only generation loop, strict Russian-label postprocessing, deduplication, progress reporting, and JSONL writing.

In [ ]:
import os, json, random, time
from dataclasses import asdict
from collections import Counter
from typing import Dict, List, Tuple, Any, Optional

from IPython.display import clear_output

COMPLEXITY_ORDER = ["L1", "L2", "L3", "L4", "L5"]

# Keep non-cinema golds only when a reliable Russian-facing label is available.

RU_GOLD_STRICT_NON_CINEMA = False  # v2-clean: keep QIDs and collect EN labels; do not drop by RU label availability
AIRPORTS_ALLOW_LABEL_FALLBACK = True
CARS_ALLOW_LABEL_FALLBACK = True
UNIVERSITIES_ALLOW_LABEL_FALLBACK = True
BOOKS_ALLOW_LABEL_FALLBACK = False
SMARTPHONES_ALLOW_LABEL_FALLBACK = True
VIDEOGAMES_ALLOW_LABEL_FALLBACK = True

def _keep_airport_fallback_label(label: str) -> Optional[str]:
    """
    Для аэропортов не режем gold слишком жёстко:
    у многих объектов нет популярного RU sitelink/label, но EN/local label вполне нормален.
    Иначе домен массово схлопывается в 0.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    return s

def _keep_car_fallback_label(label: str) -> Optional[str]:
    """
    Для моделей автомобилей разрешаем латиницу/цифры:
    Toyota Camry, BMW E39, Peugeot 508 и т.п. — это нормальные gold labels,
    даже если у сущности нет сильного RU-анкера в Wikidata.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) > 120:
        return None
    return s


def _keep_smartphone_fallback_label(label: str) -> Optional[str]:
    """
    Для смартфонов разрешаем валидные латинские названия моделей:
    iPhone 13 Pro, Galaxy S23, Pixel 8, Redmi Note 12 и т.п.
    Иначе домен схлопывается, потому что у многих моделей нет сильного RU-анкера.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 140:
        return None
    if not re.search(r"[A-Za-zА-Яа-я0-9]", s):
        return None
    return s


def _keep_videogame_fallback_label(label: str) -> Optional[str]:
    """
    Для видеоигр разрешаем хорошие латинские названия:
    Half-Life 2, Metal Gear Solid, Super Mario Odyssey, Baldur's Gate 3 и т.п.
    Иначе домен videogames после RU-нормализации легко схлопывается в 0.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 180:
        return None
    if not re.search(r"[A-Za-zА-Яа-я0-9]", s):
        return None
    return s


def _keep_university_fallback_label(label: str) -> Optional[str]:
    """
    Для университетов разрешаем fallback на исходный label:
    у многих международных университетов нет сильного RU-анкера в Wikidata,
    но EN/local label является нормальным gold answer.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 3 or len(s) > 180:
        return None
    return s


def _keep_book_fallback_label(label: str) -> Optional[str]:
    """
    Для книг fallback отключён по смыслу датасета.
    Оставляем helper только как запасной кусок кода, но в books он не должен использоваться.
    """
    if label is None:
        return None
    s = str(label).strip()
    if not s:
        return None
    if _CINEMA_BAD_LABEL_RE.fullmatch(s):
        return None
    if re.fullmatch(r"Q\d+", s):
        return None
    if len(s) < 2 or len(s) > 220:
        return None
    return s

def enforce_ru_gold_labels_non_cinema(ex: BenchmarkExample) -> None:
    if not RU_GOLD_STRICT_NON_CINEMA:
        return
    if getattr(ex, "domain", None) == "cinema":
        return

    qids = list(ex.gold_answer_qids or [])
    lbls = list(ex.gold_answer_labels_ru or [])

    if not qids:
        return

    # align lengths (defensive)
    if len(lbls) < len(qids):
        lbls = lbls + [""] * (len(qids) - len(lbls))

    items = [(qids[i], lbls[i]) for i in range(len(qids))]

    # Fetch RU terms for ALL gold qids (cached on disk via wd.cache)
    uniq_qids = list(dict.fromkeys([q for q, _ in items if q]))
    ent_map = wd_get_entities_ru(uniq_qids) if uniq_qids else {}

    seen = set()
    new_items: List[Tuple[str, str]] = []
    domain_name = getattr(ex, "domain", None)
    is_airports = (domain_name == "airports")
    is_cars = (domain_name == "cars")
    is_universities = (domain_name == "universities")
    is_books = (domain_name == "books")
    is_smartphones = (domain_name == "smartphones")
    is_videogames = (domain_name == "videogames")

    for q, l in items:
        if not q or q in seen:
            continue
        seen.add(q)

        ent = ent_map.get(q)

        best = _best_popular_ru_name(ent, fallback_label="") if ent else None

        if best:
            new_items.append((q, best))
            continue

        if is_airports and AIRPORTS_ALLOW_LABEL_FALLBACK:
            keep = _keep_airport_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_cars and CARS_ALLOW_LABEL_FALLBACK:
            keep = _keep_car_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_universities and UNIVERSITIES_ALLOW_LABEL_FALLBACK:
            keep = _keep_university_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_books and BOOKS_ALLOW_LABEL_FALLBACK:
            keep = _keep_book_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_smartphones and SMARTPHONES_ALLOW_LABEL_FALLBACK:
            keep = _keep_smartphone_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        if is_videogames and VIDEOGAMES_ALLOW_LABEL_FALLBACK:
            keep = _keep_videogame_fallback_label(l)
            if keep:
                new_items.append((q, keep))
                continue

        # If API failed for this qid, keep only if the original label already looks RU enough (has Cyrillic)
        try:
            if l and _cinema_has_cyrillic(str(l)):
                new_items.append((q, str(l).strip()))
        except Exception:
            # Drop labels that cannot be validated safely.
            pass

    ex.gold_answer_qids = [q for q, _ in new_items]
    ex.gold_answer_labels_ru = [l for _, l in new_items]

DOMAIN_GENERATORS = {
    "cinema": generate_movie_example if "generate_movie_example" in globals() else generate_cinema_example,
    "geo_ru": generate_geo_example,
    "geo": generate_geo_world_example,
    "books": generate_books_example,
    "videogames": generate_videogames_example,
    "music_albums": generate_music_albums_example,
    "software": generate_software_example,
    "people": generate_people_example_fast, 
    "scientists": generate_scientists_example,
    "physicists": generate_physicists_example,
    "mathematicians": generate_mathematicians_example,
    "paintings": generate_paintings_example,
    "museums": generate_museums_example,
    "spacecraft": generate_spacecraft_example,
    "universities": generate_universities_example,
    "airports": generate_airports_example,
    "cars": generate_cars_example,
    "countries": generate_countries_example,
    "dishes": generate_dishes_example,
    "smartphones": generate_smartphones_example,
}

TARGET_MAIN = {
    "dishes":         {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "countries":      {"L1": 5,  "L2": 12, "L3": 10, "L4": 10, "L5": 2},
    "cinema":         {"L1": 0, "L2": 0, "L3": 5, "L4": 80, "L5": 40},
    "universities":   {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "airports":       {"L1": 15,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "cars":           {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "smartphones":    {"L1": 7,  "L2": 10, "L3": 15, "L4": 7, "L5": 5},
    "paintings":      {"L1": 0,  "L2": 0, "L3": 0, "L4": 0, "L5": 15},
    "books":          {"L1": 10,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "geo_ru":         {"L1": 10,  "L2": 15, "L3": 15, "L4": 15, "L5": 10},
    "geo":            {"L1": 0,  "L2": 0, "L3": 0, "L4": 15, "L5": 10},
    "spacecraft":     {"L1": 0,  "L2":  0, "L3": 0, "L4": 10, "L5": 20},
    "museums":        {"L1": 0,  "L2": 7, "L3": 10, "L4": 10, "L5": 10},
    "people":         {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "scientists":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "physicists":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "mathematicians": {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8},
    "software":       {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
    "music_albums":   {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
    "videogames":     {"L1": 5,  "L2": 12, "L3": 15, "L4": 10, "L5": 8}, 
}

def example_fingerprint(ex: BenchmarkExample) -> str:
    payload = {
        "domain": ex.domain,
        "complexity": ex.complexity,
        "constraints": ex.constraints,
        "requested_count": ex.requested_count,
    }
    return _sha1(json.dumps(payload, ensure_ascii=False, sort_keys=True))

def _safe_iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                break

def load_existing(path: str) -> Tuple[List[BenchmarkExample], set, Counter]:
    if not os.path.exists(path):
        return [], set(), Counter()
    exs: List[BenchmarkExample] = []
    seen = set()
    cnt = Counter()
    for obj in _safe_iter_jsonl(path):
        try:
            ex = BenchmarkExample(**obj)
        except Exception:
            continue
        fp = example_fingerprint(ex)
        seen.add(fp)
        cnt[(ex.domain, ex.complexity)] += 1
        exs.append(ex)
    return exs, seen, cnt

def append_jsonl(path: str, ex: BenchmarkExample, do_fsync: bool = False):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    line = json.dumps(asdict(ex), ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        if do_fsync:
            os.fsync(f.fileno())

def min_gold_required(ex: BenchmarkExample) -> int:
    if ex.complexity in ("L1", "L2", "L3"):
        return int(ex.requested_count or 0)
    return 1

def zero_bucket_cap(domain: str, complexity: str) -> int:
    return 0

def build_main_and_zero(
    target_main: Dict[str, Dict[str, int]],
    out_main: str,
    out_zero: str,
    seed: int = 123,
    max_attempts_per_bucket: int = 2000,
    max_seconds_per_bucket: Optional[int] = 900,  
    fsync_each: bool = False,
    resume: bool = True,
    ui_every_s: float = 1.0,
):
    rng = random.Random(seed)

    if out_zero and os.path.exists(out_zero):
        os.remove(out_zero)

    if resume:
        main_ex, main_seen, main_cnt = load_existing(out_main)
    else:
        if os.path.exists(out_main):
            os.remove(out_main)
        main_ex, main_seen, main_cnt = [], set(), Counter()

    seen_all = set(main_seen)

    total_needed = sum(sum(plan.get(c, 0) for c in COMPLEXITY_ORDER) for plan in target_main.values())

    dropped_small = 0
    dropped_dup = 0
    dropped_zero = 0

    bucket_lines: Dict[str, str] = {}
    last_ui = 0.0
    global_start = time.time()

    def render(force: bool = False):
        nonlocal last_ui
        now = time.time()
        if not force and now - last_ui < ui_every_s:
            return
        last_ui = now

        done = sum(main_cnt.values())
        pct = (100.0 * done / total_needed) if total_needed else 0.0
        elapsed = int(now - global_start)

        lines = []
        lines.append(f"MAIN -> {out_main}")
        lines.append("ZERO -> disabled: empty gold lists are skipped")
        lines.append(f"MAIN total: {pct:5.1f}%  {done}/{total_needed}   elapsed={elapsed}s")
        lines.append(f"Stats: drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}")
        lines.append("")

        keys = list(bucket_lines.keys())
        tail = keys[-20:]
        for k in tail:
            lines.append(bucket_lines[k])

        clear_output(wait=True)
        print("\n".join(lines))

    render(force=True)

    for domain, plan in target_main.items():
        if domain not in DOMAIN_GENERATORS:
            continue
        gen = DOMAIN_GENERATORS[domain]

        for complexity in COMPLEXITY_ORDER:
            need_total = int(plan.get(complexity, 0))
            if need_total <= 0:
                continue

            have = main_cnt[(domain, complexity)]
            need = max(0, need_total - have)
            if need <= 0:
                continue

            bucket_key = f"{domain}:{complexity}"
            produced = 0
            attempts = 0
            start_ts = time.time()

            bucket_lines[bucket_key] = f"{bucket_key} 0/{need}  att=0  ok=0  drop_zero={dropped_zero}  drop_small={dropped_small}"
            render(force=True)

            while produced < need and attempts < max_attempts_per_bucket:
                if max_seconds_per_bucket is not None and (time.time() - start_ts) > max_seconds_per_bucket:
                    break

                attempts += 1
                bucket_lines[bucket_key] = (
                    f"{bucket_key} {produced}/{need}  att={attempts}  ok={produced}  "
                    f"drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}  "
                    f"t={int(time.time()-start_ts)}s"
                )
                render()

                idx = len(main_ex) + produced
                try:
                    ex = gen(complexity, idx, rng)
                    enforce_ru_gold_labels_non_cinema(ex)
                    ex = finalize_bilingual_clean_example(ex)
                except Exception as e:
                    if globals().get("DEBUG_GENERATOR_ERRORS"):
                        print("[GENERATOR ERROR]", domain, complexity, repr(e))
                    continue

                fp = example_fingerprint(ex)
                if fp in seen_all:
                    dropped_dup += 1
                    continue

                g = len(ex.gold_answer_qids) if isinstance(ex.gold_answer_qids, list) else 0

                if g == 0:
                    dropped_zero += 1
                    continue

                if g < min_gold_required(ex):
                    dropped_small += 1
                    continue

                append_jsonl(out_main, ex, do_fsync=fsync_each)
                main_ex.append(ex)
                main_cnt[(domain, complexity)] += 1
                seen_all.add(fp)
                produced += 1

                bucket_lines[bucket_key] = (
                    f"{bucket_key} {produced}/{need}  att={attempts}  ok={produced}  "
                    f"drop_zero={dropped_zero}  drop_small={dropped_small}  drop_dup={dropped_dup}  "
                    f"t={int(time.time()-start_ts)}s"
                )
                render()

            if produced < need:
                extra = ""
                if max_seconds_per_bucket is not None and (time.time() - start_ts) > max_seconds_per_bucket:
                    extra = f", time_limit={max_seconds_per_bucket}s"
                bucket_lines[bucket_key] = f"[WARN] MAIN {bucket_key} need={need}, produced={produced} (attempts={attempts}{extra})"
                render(force=True)

    render(force=True)
    print("\nDone.")
    print("MAIN saved:", out_main, "count=", len(main_ex))
    print("ZERO disabled; skipped empty gold examples:", dropped_zero)
    return main_ex, []


main_examples, zero_examples = build_main_and_zero(
    TARGET_MAIN,
    out_main=DATASET_PATH_MAIN,
    out_zero=DATASET_PATH_ZERO,
    seed=123,
    max_attempts_per_bucket=25000,
    max_seconds_per_bucket=20000,  
    fsync_each=False,
    resume=True,
    ui_every_s=0.7,
)


MAIN -> out_wikidata_benchmark/dataset_main.jsonl
ZERO -> disabled: empty gold lists are skipped
MAIN total: 108.5%  1080/995   elapsed=0s
Stats: drop_zero=25000  drop_small=0  drop_dup=0

[WARN] MAIN cinema:L4 need=14, produced=0 (attempts=25000)
cinema:L5 0/6  att=0  ok=0  drop_zero=25000  drop_small=0
[1080 | Поп. 1/25] L5: Кристиан Бэйл, драма, 1-90 мин, imdb>=7.0 ... Ошибка API: 403
[1080 | Поп. 2/25] L5: Леонардо ДиКаприо, драма, 1-180 мин, imdb>=7.5 ... Ошибка API: 403
[1080 | Поп. 3/25] L5: Леонардо ДиКаприо, фантастика, 1-120 мин, imdb>=6.5 ... 

KeyboardInterrupt: 

<a id="summary"></a>

## 26. Summary

The notebook defines a reusable Wikidata-based pipeline for generating Russian multihop benchmark examples across multiple domains. It keeps the shared WDQS/cache utilities, domain-specific generators, Russian-label postprocessing, deduplication, and append-only JSONL assembly in separate sections. The expected output is a main dataset file with non-zero-gold examples that can be manually validated and then used for LLM evaluation.